# German–Turkish parallel dative examples (UD German-PUD)

This notebook reads the parallel UD German-PUD treebank (`de_pud-ud-test.conllu` / `tr_pud-ud-test.conllu`), finds German dative constructions for the preposition categories defined in `testset/dative_categories/` (`dative_prep_in`, `dative_prep_zu`, …), and prints up to **10 sentence pairs per category**.

For each German dative span (`group_span_text`), it also attempts to locate the corresponding Turkish span (`turkish_group_span_text`) using UD case labels, shared proper nouns/numbers, and small lemma translation hints (`pud_turkish_dative_align.py`).

Classification reuses the same logic as the test-set pipeline (`ud_corruption_utils.classify_dative_type`).

In [21]:
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional

import pandas as pd

from load_ud_dataset import parse_conllu_file
from pud_turkish_dative_align import (
    collect_span_text,
    find_turkish_dative_span,
)
from ud_corruption_utils import (
    case_marker_lemma_for_head_id,
    classify_dative_type,
    normalize_case_prep_lemma,
)

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
PUD_DIR = REPO_ROOT / "data" / "UD_German-PUD"
CATEGORY_DIR = REPO_ROOT / "UD_Dataset" / "testset" / "dative_categories"

DE_CONLLU = PUD_DIR / "de_pud-ud-test.conllu"
TR_CONLLU = PUD_DIR / "tr_pud-ud-test.conllu"

MAX_PER_CATEGORY = 10

TARGET_CATEGORIES = [
    "dative_prep_in",
    "dative_prep_zu",
    "dative_prep_mit",
    "dative_prep_von",
    "dative_prep_an",
    "dative_prep_nach",
    "dative_prep_bei",
    "dative_prep_aus",
    "dative_prep_auf",
    "dative_prep_vor",
    "dative_prep_unter",
    "dative_prep_seit",
]

DAT_POS = {"NOUN", "PROPN", "PRON"}

In [22]:
def load_parallel_pud(de_path: Path, tr_path: Path) -> tuple[list[dict[str, Any]], dict[str, dict[str, Any]]]:
    """Load German sentences and index Turkish sentences by ``sent_id``."""
    de_sents = parse_conllu_file(str(de_path))
    tr_sents = parse_conllu_file(str(tr_path))
    tr_by_id = {s["sent_id"]: s for s in tr_sents if s.get("sent_id")}
    print(f"German: {len(de_sents)} sentences")
    print(f"Turkish: {len(tr_sents)} sentences")
    print(f"Aligned by sent_id: {sum(1 for s in de_sents if s.get('sent_id') in tr_by_id)}")
    return de_sents, tr_by_id


def find_dative_hits(
    de_sents: list[dict[str, Any]],
    categories: list[str],
) -> dict[str, list[dict[str, Any]]]:
    """Return up to one hit per German sentence for each target category."""
    hits: dict[str, list[dict[str, Any]]] = {c: [] for c in categories}
    seen: dict[str, set[str]] = {c: set() for c in categories}

    for sent in de_sents:
        sent_id = sent.get("sent_id", "")
        tokens = sent["tokens"]
        for i, tok in enumerate(tokens):
            if tok["upos"] not in DAT_POS:
                continue
            if tok["feats"].get("Case") != "Dat":
                continue

            case_lemma = case_marker_lemma_for_head_id(tokens, tok["id"])
            dative_type = classify_dative_type(
                tok,
                case_lemma,
                tokens=tokens,
                head_idx=i,
            )
            if dative_type not in categories or sent_id in seen[dative_type]:
                continue

            seen[dative_type].add(sent_id)
            hits[dative_type].append(
                {
                    "sent_id": sent_id,
                    "dative_type": dative_type,
                    "preposition": normalize_case_prep_lemma(case_lemma),
                    "de_head_id": tok["id"],
                    "group_span_text": collect_span_text(tokens, tok["id"]),
                    "gold_text": sent.get("text", ""),
                }
            )
    return hits


def attach_turkish(
    hits: dict[str, list[dict[str, Any]]],
    de_by_id: dict[str, dict[str, Any]],
    tr_by_id: dict[str, dict[str, Any]],
    max_per_category: int,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for category in TARGET_CATEGORIES:
        for hit in hits[category][:max_per_category]:
            sid = hit["sent_id"]
            de_sent = de_by_id.get(sid)
            tr_sent = tr_by_id.get(sid)
            tr_span, tr_score, tr_head = (
                find_turkish_dative_span(
                    de_sent,
                    tr_sent,
                    de_head_id=hit["de_head_id"],
                    prep=hit["preposition"],
                )
                if de_sent and tr_sent
                else (None, 0, None)
            )
            rows.append(
                {
                    **hit,
                    "turkish_text": tr_sent.get("text", "") if tr_sent else "",
                    "turkish_group_span_text": tr_span,
                    "turkish_head_form": tr_head,
                    "turkish_alignment_score": tr_score,
                }
            )
    return pd.DataFrame(rows)


def print_parallel_pairs(df: pd.DataFrame, category: str) -> None:
    subset = df[df["dative_type"] == category]
    print(f"\n{'=' * 80}")
    print(f"{category}  ({len(subset)} examples)")
    print('=' * 80)
    if subset.empty:
        print("  (no examples found)")
        return
    for n, row in enumerate(subset.itertuples(), start=1):
        print(
            f"\n[{n}] sent_id={row.sent_id}  prep={row.preposition}  "
            f"de_span={row.group_span_text!r}  tr_span={row.turkish_group_span_text!r}"
        )
        print(f"  DE: {row.gold_text}")
        print(f"  TR: {row.turkish_text}")

In [27]:
de_sents, tr_by_id = load_parallel_pud(DE_CONLLU, TR_CONLLU)
de_by_id = {sent["sent_id"]: sent for sent in de_sents if sent.get("sent_id")}
hits = find_dative_hits(de_sents, TARGET_CATEGORIES)

summary = pd.DataFrame(
    [
        {
            "category": cat,
            "preposition": cat.removeprefix("dative_prep_"),
            "sentences_in_pud": len(hits[cat]),
            "selected": min(len(hits[cat]), MAX_PER_CATEGORY),
        }
        for cat in TARGET_CATEGORIES
    ]
)
summary

German: 1000 sentences
Turkish: 1000 sentences
Aligned by sent_id: 1000


,category,preposition,sentences_in_pud,selected
0,dative_prep_in,in,380,10
1,dative_prep_zu,zu,143,10
2,dative_prep_mit,mit,124,10
3,dative_prep_von,von,237,10
4,dative_prep_an,an,89,10
5,dative_prep_nach,nach,57,10
6,dative_prep_bei,bei,51,10
7,dative_prep_aus,aus,42,10
8,dative_prep_auf,auf,46,10
9,dative_prep_vor,vor,19,10


In [28]:
parallel_df = attach_turkish(hits, de_by_id, tr_by_id, MAX_PER_CATEGORY)
parallel_df[[
    "dative_type",
    "preposition",
    "sent_id",
    "group_span_text",
    "turkish_group_span_text",
    "turkish_alignment_score",
    "gold_text",
    "turkish_text",
]].head(5)

,dative_type,preposition,sent_id,group_span_text,turkish_group_span_text,turkish_alignment_score,gold_text,turkish_text
0,dative_prep_in,in,n01001011,einem Blogeintrag,blog yazısında,106,„Ein Großteil des digitalen Übergangs ist für ...,Obama'nın Özel Asistanı Kori Schulman Pazartes...
1,dative_prep_in,in,n01002032,Händen,sizin elinizde,85,"„Ich möchte euch ungern unter Druck setzen, ab...",North Carolina Üniversitesi'ndeki sahada topla...
2,dative_prep_in,in,n01003012,dem exklusiven Metropolitan Club,Club,281,Das Treffen sollte ursprünglich nur ein paar B...,"Buluşma yeri esasen Washington, H Street üzeri..."
3,dative_prep_in,in,n01004017,der 4 Klasse,Dördüncü sınıflar arasında,17,In der 4. Klasse liegen sie im nationalen Durc...,Dördüncü sınıflar arasında milli ortalamayla a...
4,dative_prep_in,in,n01005023,dem Vergleich,2004'te,23,"Im Vergleich dazu kostete es 103,7 Millionen D...","Karşılaştıracak olursak, 2004'te açılan NoMa m..."


In [29]:
for category in TARGET_CATEGORIES:
    print_parallel_pairs(parallel_df, category)


dative_prep_in  (10 examples)

[1] sent_id=n01001011  prep=in  de_span='einem Blogeintrag'  tr_span='blog yazısında'
  DE: „Ein Großteil des digitalen Übergangs ist für die Vereinigten Staaten neu, ein friedlicher Machtwechsel hingegen nicht“, schrieb Obamas Sonderberaterin Kori Schulman am Montag in einem Blogeintrag.
  TR: Obama'nın Özel Asistanı Kori Schulman Pazartesi günü yayınladığı blog yazısında "ABD'deki dijital dönüşümün büyük bir kısmı emsalsiz olsa da, başkanlık görevinin sorunsuz şekilde devredildiği görülmemiştir" dedi.

[2] sent_id=n01002032  prep=in  de_span='Händen'  tr_span='sizin elinizde'
  DE: „Ich möchte euch ungern unter Druck setzen, aber das Schicksal der Republik liegt in euren Händen“, sagte er zu der Menschenmenge, die sich auf einem Sportplatz an der University of North Carolina versammelt hatte.
  TR: North Carolina Üniversitesi'ndeki sahada toplanmış kalabalığa "Size baskı yapmak istemem ama halkın kaderi sizin elinizde." dedi.

[3] sent_id=n01003012  pr

In [30]:
# Optional: save the sampled pairs for downstream use
output_path = NOTEBOOK_DIR / "pud_dative_parallel_samples.json"
parallel_df.to_json(output_path, orient="records", force_ascii=False, indent=2)
print(f"Saved {len(parallel_df)} rows to {output_path}")

Saved 120 rows to /Users/ebrark/Desktop/masters🎓/HiWi Working/github/Morphology/UD_Dataset/pud_dative_parallel_samples.json


In [32]:
table_path = NOTEBOOK_DIR / "pud_dative_parallel_table.csv"
table_columns = [
    "sent_id",
    "preposition",
    "DE span",
    "TR span",
    "DE gold text",
    "TR gold text",
]

table_lines = ["\t".join(table_columns)]
for _, row in parallel_df.iterrows():
    table_lines.append(
        "\t".join(
            [
                row["sent_id"],
                row["preposition"],
                row["group_span_text"] or "",
                row.get("turkish_group_span_text") or "",
                row["gold_text"],
                row["turkish_text"],
            ]
        )
    )

table_path.write_text("\n".join(table_lines) + "\n", encoding="utf-8")
print(f"Saved tab-separated table to {table_path}")
print("\n".join(table_lines[:3]))

Saved tab-separated table to /Users/ebrark/Desktop/masters🎓/HiWi Working/github/Morphology/UD_Dataset/pud_dative_parallel_table.csv
sent_id	preposition	DE span	TR span	DE gold text	TR gold text
n01001011	in	einem Blogeintrag	blog yazısında	„Ein Großteil des digitalen Übergangs ist für die Vereinigten Staaten neu, ein friedlicher Machtwechsel hingegen nicht“, schrieb Obamas Sonderberaterin Kori Schulman am Montag in einem Blogeintrag.	Obama'nın Özel Asistanı Kori Schulman Pazartesi günü yayınladığı blog yazısında "ABD'deki dijital dönüşümün büyük bir kısmı emsalsiz olsa da, başkanlık görevinin sorunsuz şekilde devredildiği görülmemiştir" dedi.
n01002032	in	Händen	sizin elinizde	„Ich möchte euch ungern unter Druck setzen, aber das Schicksal der Republik liegt in euren Händen“, sagte er zu der Menschenmenge, die sich auf einem Sportplatz an der University of North Carolina versammelt hatte.	North Carolina Üniversitesi'ndeki sahada toplanmış kalabalığa "Size baskı yapmak istemem ama halkın